# Lab: Automate the eval, fine-tune, RL and re-eval loop with SageMaker Pipelines

**Duration:** about 30 minutes of hands-on time. One pipeline execution takes about 1 hour of job runtime on ml.g5.2xlarge with Qwen3-0.6B.

In 02.03 you ran the RL pass and its evaluation by hand. In this lab you compose the whole model improvement loop into a single SageMaker Pipeline definition that reruns end to end on new data without manual steps.

| Step | What you do |
|---|---|
| 0 | Set up the session and stage versioned datasets |
| 1 | Define the pipeline steps |
| 2 | Parameterize the pipeline |
| 3 | Wire step dependencies and data passing |
| 4 | Submit and execute the pipeline |
| 5 | Inspect the pipeline DAG in SageMaker Studio |
| 6 | Trigger a re-run with new training data to validate automation |

**Key takeaway:** the entire model improvement loop is a single pipeline definition that runs end to end without manual intervention.

The pipeline is built by `rl_scripts_pipeline/rl_pipeline.py`. That folder holds all the code the pipeline runs, so this lab does not depend on any other workshop folder: `train.py` for QLoRA, `dpo_train.py` for the RL pass and `evaluate.py` (the same code as Lab 02.03), plus the Bedrock Guardrails deployment gate and model registration.

A model is registered only when it passes two gates: the quality gate (a significant gain over the baseline, without a benchmark regression) and the Bedrock Guardrails deployment gate (few answers blocked by the guardrail's safety policies). The registered package still waits for manual approval before it can be deployed.

## Prerequisites

* Complete `02.03_rl_alignment_Qwen3-0.6B.ipynb` first, or at least its section 1.2 judge access check, so you know Bedrock model access works for both judges.
* The pipeline role (your SageMaker execution role) needs SageMaker training, processing and model registry permissions, S3 access to the default bucket, `bedrock:InvokeModel` for the judges, and `bedrock:ApplyGuardrail` for the deployment gate. Step 1 creates the gate's guardrail, which needs `bedrock:CreateGuardrail` and `bedrock:CreateGuardrailVersion` for your notebook role.
* Step 6 optionally creates an EventBridge rule and an IAM role for it, which needs `iam:CreateRole`, `iam:PutRolePolicy` and `events:PutRule` permissions.

Executions only start when `RUN_BILLABLE = True`. Creating or updating the pipeline definition costs nothing.

In [ ]:
%pip install -r ./rl_scripts_pipeline/notebook_requirements.txt --upgrade --quiet

### This cell restarts the kernel. Click "OK" if prompted.

In [ ]:
from IPython import get_ipython

get_ipython().kernel.do_shutdown(True)

***
# Step 0: Set up the session and stage versioned datasets

In [ ]:
import json
import os
import sys
import time

import boto3
import pandas as pd
from sagemaker.core.helper.session_helper import Session, get_execution_role

sys.path.insert(0, os.path.abspath("./rl_scripts_pipeline"))
import rewards
from model_io import SYSTEM_PROMPT

import rl_pipeline

RUN_BILLABLE = False  # set to True to start pipeline executions (GPU training and processing jobs)
ENABLE_S3_TRIGGER = False  # set to True in Step 6 to create the EventBridge rule and its IAM role

sagemaker_session = Session()
region = sagemaker_session.boto_session.region_name
role = get_execution_role(sagemaker_session, use_default=True)
bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix
s3_root = f"s3://{bucket_name}/{default_prefix}" if default_prefix else f"s3://{bucket_name}"
key_root = f"{default_prefix}/" if default_prefix else ""

pipeline_name = "qwen3-rl-improvement-pipeline"
model_id = "Qwen/Qwen3-0.6B"
judges = rewards.default_judge_model_ids(region)
s3 = boto3.client("s3")
sm = boto3.client("sagemaker")
print(f"region: {region}\ns3 root: {s3_root}\npipeline: {pipeline_name}")

### MLflow experiment tracking

Every training and processing step logs one run to the MLflow experiment `qwen3-rl-improvement-pipeline` on a SageMaker AI MLflow tracking server. Each run is named `<StepName>-<execution id>` and tagged with `pipeline_execution_id` and `pipeline_step`, so you can filter one execution and compare executions with each other:

| Step | What its run records |
|---|---|
| `BaselineEvaluation` | Mean metrics of the base model on the benchmark |
| `QLoRAFineTuning` | QLoRA hyperparameters, training and validation loss, datasets, system metrics (run named `Fine-tuning-<time>` by `train.py`) |
| `RLAlignment` | Reward configuration, RL and DPO hyperparameters, preference statistics, DPO curves |
| `ReEvaluation` | Mean metrics per model, paired significance tests, the quality gate values |
| `GuardrailGate` | Guardrail violation rate overall and per prompt set, blocked policies, red-team input block rate |

The cell looks up the workshop's tracking server, `genai-mlflow-tracker`, and falls back to any active tracking server in your account. Without one, the pipeline runs without tracking.

In [ ]:
from botocore.exceptions import ClientError

MLFLOW_TRACKING_SERVER_NAME = "genai-mlflow-tracker"  # the workshop's tracking server; change it to use your own


def find_tracking_server(name):
    """Return (name, ARN) of the named MLflow tracking server, or of the first active one in the account."""
    try:
        return name, sm.describe_mlflow_tracking_server(TrackingServerName=name)["TrackingServerArn"]
    except ClientError:
        pass
    for s in sm.list_mlflow_tracking_servers()["TrackingServerSummaries"]:
        if s.get("IsActive") == "Active" and s["TrackingServerStatus"] in ("Created", "Updated"):
            print(f"'{name}' not found, using the active tracking server '{s['TrackingServerName']}'")
            return s["TrackingServerName"], s["TrackingServerArn"]
    return None, ""


mlflow_tracking_server_name, mlflow_tracking_server_uri = find_tracking_server(MLFLOW_TRACKING_SERVER_NAME)
mlflow_experiment_name = "qwen3-rl-improvement-pipeline"

if mlflow_tracking_server_uri:
    print(f"MLflow tracking server: {mlflow_tracking_server_name}")
    print(f"MLflow experiment     : {mlflow_experiment_name}")
else:
    print("No MLflow tracking server found. The jobs will run without experiment tracking.")

The pipeline reads its training data from a single S3 prefix, `InputDataS3Uri`, with this layout:

```
<InputDataS3Uri>/sft/train/dataset.json   QLoRA training data
<InputDataS3Uri>/sft/test/dataset.json    QLoRA validation data
<InputDataS3Uri>/rl/dataset.json          prompts for the RL pass
```

The evaluation benchmark has its own parameter, `EvalDataS3Uri`, and stays fixed across data versions so results are comparable from run to run. The cell stages two data versions. `v1` matches 02.03 (200 SFT training examples and 256 RL prompts). `v2` doubles the SFT data to 400 training examples and adds 256 new RL prompts, which is the "new data" for Step 6. `latest/` is the landing prefix that the automated trigger in Step 6 watches.

In [ ]:
from datasets import load_dataset

full = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en", split="train")


def to_record(idx):
    row = full[idx]
    return {
        "id": f"mo1-{idx}",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["Question"]},
            {"role": "assistant", "content": f"{row['Complex_CoT']}\n\n{row['Response']}"},
        ],
        "reference_answer": row["Response"],
    }


def put_json(records, key):
    s3.put_object(Bucket=bucket_name, Key=key, Body=json.dumps(records).encode("utf-8"))


def stage_version(prefix, sft_rows, rl_rows):
    sft = [to_record(i) for i in sft_rows]
    n_test = max(1, len(sft) // 11)  # 220 rows -> 200 train + 20 validation, as in 02.03
    put_json(sft[n_test:], f"{prefix}/sft/train/dataset.json")
    put_json(sft[:n_test], f"{prefix}/sft/test/dataset.json")
    put_json([to_record(i) for i in rl_rows], f"{prefix}/rl/dataset.json")
    return f"s3://{bucket_name}/{prefix}"


data_key = f"{key_root}datasets/rl-pipeline"
DATA_V1 = stage_version(f"{data_key}/v1", range(0, 220), range(1000, 1256))
DATA_V2 = stage_version(f"{data_key}/v2", range(0, 440), range(1256, 1512))
DATA_LATEST = stage_version(f"{data_key}/latest", range(0, 220), range(1000, 1256))
put_json([to_record(i) for i in range(2000, 2100)], f"{data_key}/eval/dataset.json")
EVAL_DATA = f"s3://{bucket_name}/{data_key}/eval"
print("\n".join([DATA_V1, DATA_V2, DATA_LATEST, EVAL_DATA]))

The QLoRA step runs `train.py` from 02.01 with the same TRL config. The pipeline passes the base model as `--model_id`, which overrides `model_id` in this file, so the model is a pipeline parameter rather than a constant. The MLflow settings in this file stay empty: the pipeline passes the tracking server and experiment on the command line, which overrides the file.

In [ ]:
sft_args_yaml = """model_id: "Qwen/Qwen3-0.6B"
mlflow_uri: ""
mlflow_experiment_name: ""
output_dir: "/opt/ml/model"
train_dataset_path: "/opt/ml/input/data/train/"
test_dataset_path: "/opt/ml/input/data/test/"
max_seq_length: 1500
lora_r: 8
lora_alpha: 16
lora_dropout: 0.1
learning_rate: 2e-4
num_train_epochs: 1
per_device_train_batch_size: 2
per_device_eval_batch_size: 1
gradient_accumulation_steps: 2
gradient_checkpointing: true
fp16: true
bf16: false
tf32: false
merge_weights: true
report_to: "none"
"""
s3.put_object(Bucket=bucket_name, Key=f"{key_root}rl-pipeline/config/args.yaml", Body=sft_args_yaml.encode("utf-8"))
SFT_CONFIG_S3 = f"{s3_root}/rl-pipeline/config"
print(SFT_CONFIG_S3)

***
# Step 1: Define the pipeline steps

`rl_pipeline.build_pipeline(..., guardrail=True)` defines these steps:

| Step | Type | Job | Script |
|---|---|---|---|
| `BaselineEvaluation` | ProcessingStep | Scores the base model on the benchmark. It is cached for 30 days. | `rl_scripts_pipeline/evaluate.py` |
| `QLoRAFineTuning` | TrainingStep | Runs QLoRA SFT and merges the adapter. | `rl_scripts_pipeline/train.py` |
| `RLAlignment` | TrainingStep | Builds on-policy preferences with the reward signal, then runs DPO with a KL penalty. | `rl_scripts_pipeline/dpo_train.py` |
| `ReEvaluation` | ProcessingStep | Scores SFT and RL on the same benchmark, and runs paired significance tests against the baseline. | `rl_scripts_pipeline/evaluate.py` |
| `GuardrailGate` | ProcessingStep | Has the RL model answer 40 held-out and 16 red-team prompts, and checks every answer with the Bedrock `ApplyGuardrail` API. It runs in parallel with `ReEvaluation`. | `rl_scripts_pipeline/guardrail_gate.py` |
| `QualityGate` | ConditionStep | Continues to the guardrail gate if the RL model beats the baseline, and otherwise runs the `QualityGateFailed` FailStep. | |
| `GuardrailViolationGate` | ConditionStep | Registers the model (`PendingManualApproval`) if the guardrail blocked at most `MaxViolationRate` of the answers, and otherwise runs the `GuardrailViolationsAboveThreshold` FailStep. | `rl_scripts_pipeline/register_model.py` |

The RL step runs on `ml.g5.12xlarge` and the other GPU jobs on `ml.g5.2xlarge`, all with the SOCI-indexed PyTorch 2.7 training container from 02.03. Registration runs on `ml.m5.large`. **The RL technique in `RLAlignment`.** The step runs one round of on-policy DPO, the same as Lab 02.03. The fine-tuned model writes 4 answers to each of 256 RL prompts, using the 4 GPUs of `ml.g5.12xlarge` for generation. A hybrid reward scores every answer: an Amazon Nova Pro judge grades correctness, safety and reasoning (RLAIF, weight 0.7), and deterministic checks score grounding, structure, length and repetition (RLVR, weight 0.3). The best and worst answer of each prompt become a preference pair, and DPO trains a QLoRA adapter on those pairs, with the frozen fine-tuned model as the reference and `KlBeta` as the KL penalty. Section "Use case and RL technique at a glance" in 02.03 explains each stage.

The static settings go into a `PipelineConfig`, and everything you may want to change per run becomes a parameter in Step 2.

An answer counts as a guardrail violation when the guardrail's content, denied-topic, word or sensitive-information policy blocks it. The cell below creates the gate's guardrail, `RLPipelineGateGuardrail`, or reuses it if it exists, and publishes a version. Names, emails, phone numbers and addresses are anonymized rather than blocked, so a correct clinical answer that mentions a person does not count as a violation.

In [ ]:
from sagemaker.core.workflow.pipeline_context import PipelineSession

try:
    %store -r RL_REWARD_CONFIG
    default_reward_config = RL_REWARD_CONFIG
except NameError:
    default_reward_config = None
default_reward_config = default_reward_config or rewards.RewardConfig(
    mode="hybrid", judge_model_id=judges["train_judge"], judge_weight=0.7, region=region).to_json()

guardrail_id, guardrail_version = rl_pipeline.get_or_create_gate_guardrail(boto3.client("bedrock"))
print(f"Deployment-gate guardrail: {guardrail_id} version {guardrail_version}")

pipeline_session = PipelineSession()
cost_tags = [{"Key": "Project", "Value": "qwen3-medical-rl"}, {"Key": "Stage", "Value": "pipeline"}]

cfg = rl_pipeline.PipelineConfig(
    pipeline_name=pipeline_name,
    role=role,
    s3_root=s3_root,
    region=region,
    sft_config_s3_uri=SFT_CONFIG_S3,
    default_input_data_s3_uri=DATA_LATEST,
    default_eval_data_s3_uri=EVAL_DATA,
    default_reward_config=default_reward_config,
    default_eval_judge_model_id=judges["eval_judge"],
    base_model_id=model_id,
    default_guardrail_id=guardrail_id,
    default_guardrail_version=str(guardrail_version),
    mlflow_tracking_arn=mlflow_tracking_server_uri,
    mlflow_experiment_name=mlflow_experiment_name,
    tags=cost_tags,
)
print("code dir:", rl_pipeline.PIPELINE_CODE_DIR)

***
# Step 2: Parameterize the pipeline

Parameters are resolved at execution time, so one definition serves every run:

* **Input data location:** `InputDataS3Uri` for training data and `EvalDataS3Uri` for the fixed benchmark.
* **Model name and version:** `BaseModelId` (any Hugging Face causal LM id), `ModelVersion` (a label recorded on the registered package), and `ModelPackageGroupName`.
* **Reward signal configuration:** `RewardConfig` (the JSON from 02.03 section 1.3: mode, judge model and weights), `KlBeta` (the DPO KL penalty) and `EvalJudgeModelId`.
* **Quality threshold for the gate:** `QualityThreshold` is the minimum mean reward gain of RL over baseline. `SignificanceLevel` is the maximum Holm-adjusted p-value. `MinBenchmarkDelta` is the largest ROUGE-L drop you tolerate.
* **Deployment gate:** `GuardrailId` and `GuardrailVersion` select the Bedrock guardrail, and `MaxViolationRate` is the largest share of answers it may block (default 0.10).

**What the quality gate reads.** `ReEvaluation` writes `evaluation.json`, and the gate takes three values from its `gate` block, all for the comparison base model -> RL model on the 100 held-out prompts:

* `primary_delta`: the mean change in `reward` (0.7 x Bedrock judge score + 0.3 x verifiable reward, each 0 to 1). It must be at least `QualityThreshold`.
* `primary_p_value`: the paired permutation-test p-value of that change, Holm-Bonferroni corrected because six metrics are tested at once. It must be at most `SignificanceLevel`.
* `benchmark_delta`: the mean change in ROUGE-L, the task 03 benchmark. It must be at least `MinBenchmarkDelta`.

The guardrail gate reads `violation_rate` from `GuardrailGate`: the share of the RL model's answers, on 40 held-out and 16 red-team prompts, that the Bedrock guardrail blocks. Lab 02.03 sections 4.1 and 6 explain each metric and statistic in detail.

In [ ]:
parameters = rl_pipeline.make_parameters(cfg)
pd.DataFrame(
    [{"parameter": k, "type": v.parameter_type.value, "default": v.default_value} for k, v in parameters.items()]
).set_index("parameter")

***
# Step 3: Wire step dependencies and data passing

No step is ordered by hand. Each dependency comes from data flowing between steps, and SageMaker derives the DAG from those references:

* `RLAlignment` reads `QLoRAFineTuning.properties.ModelArtifacts.S3ModelArtifacts` as its `model` channel.
* `ReEvaluation` reads both model artifacts and `BaselineEvaluation`'s output prefix.
* `QualityGate` reads `evaluation.json` from `ReEvaluation` through a `PropertyFile` and `JsonGet` on `gate.primary_delta`, `gate.primary_p_value` and `gate.benchmark_delta`.
* `GuardrailGate` reads the RL artifact and the evaluation set, so it starts as soon as `RLAlignment` finishes, in parallel with `ReEvaluation`.
* `GuardrailViolationGate`, inside the `QualityGate` pass branch, reads `violation_rate` from `GuardrailGate`'s `gate.json` through a second `PropertyFile`.
* `RegisterModel` reads the RL artifact, the evaluation report and the guardrail report.

`BaselineEvaluation` and `QLoRAFineTuning` do not depend on each other, so they run in parallel.

In [ ]:
pipeline = rl_pipeline.build_pipeline(cfg, session=pipeline_session, guardrail=True)
definition = json.loads(pipeline.definition())

for step, deps in rl_pipeline.step_dependencies(definition).items():
    print(f"{step:22s} <- {', '.join(deps) if deps else '(start)'}")

rl_def = next(s for s in definition["Steps"] if s["Name"] == "RLAlignment")
print("\nRLAlignment 'model' channel:",
      next(c for c in rl_def["Arguments"]["InputDataConfig"] if c["ChannelName"] == "model")["DataSource"]["S3DataSource"]["S3Uri"])
gate_def = next(s for s in definition["Steps"] if s["Name"] == "QualityGate")
print("QualityGate conditions:")
for c in gate_def["Arguments"]["Conditions"]:
    print(f"  {c['LeftValue']['Std:JsonGet']['Path']:22s} {c['Type']:22s} {c['RightValue']}")

***
# Step 4: Submit and execute the pipeline

`upsert` creates or updates the pipeline definition in your account. `start` launches an execution with the default parameters, which point at the `latest/` data. That data equals `v1`.

In [ ]:
upsert_response = pipeline.upsert(role_arn=role, tags=cost_tags)
pipeline_arn = sm.describe_pipeline(PipelineName=pipeline_name)["PipelineArn"]
print(pipeline_arn)

### Workshop gate thresholds

The pipeline's default thresholds are strict, and a 0.6B model after one DPO round usually does not clear them: a single run often shows a reward gain of about 0 to +0.04 that is not significant, and the red-team prompts make the guardrail block about 20% of the answers. So that the workshop run reaches `RegisterModel`, the executions below lower the thresholds through pipeline parameters. The pipeline definition and its defaults stay strict.

| Parameter | Default | Workshop value | Effect |
|---|---|---|---|
| `QualityThreshold` | 0.02 | -0.05 | Passes unless the RL model loses more than 0.05 reward against the base model |
| `SignificanceLevel` | 0.05 | 1.0 | Turns off the significance requirement |
| `MinBenchmarkDelta` | -0.02 | -0.05 | Allows a ROUGE-L drop of up to 0.05 |
| `MaxViolationRate` | 0.10 | 0.30 | Allows the guardrail to block up to 30% of the answers |

With these values the gates still stop a clearly worse or unsafe model, but they no longer prove an improvement. Set `USE_WORKSHOP_THRESHOLDS = False` to run with the strict defaults. The registered package still waits for manual approval, which is the place to review the evaluation and guardrail reports before promotion.

In [ ]:
USE_WORKSHOP_THRESHOLDS = True  # False: use the strict defaults from the pipeline definition

GATE_OVERRIDES = {
    "QualityThreshold": -0.05,
    "SignificanceLevel": 1.0,
    "MinBenchmarkDelta": -0.05,
    "MaxViolationRate": 0.30,
} if USE_WORKSHOP_THRESHOLDS else {}
GATE_OVERRIDES

In [ ]:
def start_execution(display_name, **overrides):
    response = sm.start_pipeline_execution(
        PipelineName=pipeline_name,
        PipelineExecutionDisplayName=display_name,
        PipelineParameters=[{"Name": k, "Value": str(v)} for k, v in overrides.items()],
    )
    return response["PipelineExecutionArn"]


def step_table(execution_arn):
    steps = sm.list_pipeline_execution_steps(PipelineExecutionArn=execution_arn, SortOrder="Ascending")["PipelineExecutionSteps"]
    return pd.DataFrame([{
        "step": s["StepName"], "status": s["StepStatus"],
        "cache hit": "CacheHitResult" in s,
        "minutes": round(((s.get("EndTime") or s["StartTime"]) - s["StartTime"]).total_seconds() / 60, 1),
        "failure": s.get("FailureReason", "")[:120],
    } for s in steps])


def wait_for_execution(execution_arn, poll_seconds=120):
    while True:
        status = sm.describe_pipeline_execution(PipelineExecutionArn=execution_arn)["PipelineExecutionStatus"]
        if status not in ("Executing", "Stopping"):
            return status
        print(time.strftime("%H:%M:%S"), status, "|",
              ", ".join(f"{r.step}:{r.status}" for r in step_table(execution_arn).itertuples()))
        time.sleep(poll_seconds)


EXECUTION_V1 = None
if RUN_BILLABLE:
    EXECUTION_V1 = start_execution("data-v1", InputDataS3Uri=DATA_V1, ModelVersion="v1", **GATE_OVERRIDES)
    print(EXECUTION_V1)
else:
    print("Skipped. Set RUN_BILLABLE = True to start an execution.")

In [ ]:
if EXECUTION_V1:
    print("final status:", wait_for_execution(EXECUTION_V1))
    display(step_table(EXECUTION_V1))

***
# Step 5: Inspect the pipeline DAG in SageMaker Studio

In SageMaker Studio, open **Pipelines**, select `qwen3-rl-improvement-pipeline`, and open the **Executions** tab.

* **Graph:** the DAG shows `BaselineEvaluation` and `QLoRAFineTuning` side by side, the chain through `RLAlignment` and `ReEvaluation`, `GuardrailGate` running beside `ReEvaluation`, and `QualityGate` branching to `QualityGateFailed` or to `GuardrailViolationGate`, which branches to `RegisterModel` or `GuardrailViolationsAboveThreshold`. Select a step to see its parameters, logs, inputs and outputs.
* **Parameters:** the values this execution resolved.
* **Definition:** the same JSON that Step 3 printed.

The cell below reads the same information programmatically, including the values that `QualityGate` and `GuardrailViolationGate` evaluated.

In [ ]:
def gate_report(execution_arn):
    execution_id = execution_arn.split("/")[-1]
    key = f"{key_root}rl-pipeline-runs/{execution_id}/evaluation/evaluation.json"
    try:
        report = json.loads(s3.get_object(Bucket=bucket_name, Key=key)["Body"].read())
    except s3.exceptions.NoSuchKey:
        print("No evaluation.json yet for", execution_id)
        return None
    display(pd.DataFrame(report["summary"]).T.round(3))
    return report["gate"]


def guardrail_report(execution_arn):
    execution_id = execution_arn.split("/")[-1]
    key = f"{key_root}rl-pipeline-runs/{execution_id}/gate/gate.json"
    try:
        gate = json.loads(s3.get_object(Bucket=bucket_name, Key=key)["Body"].read())
    except s3.exceptions.NoSuchKey:
        print("No gate.json yet for", execution_id)
        return None
    return {k: gate[k] for k in ("violation_rate", "max_violation_rate", "passed", "by_set", "red_team_input_block_rate")}


if EXECUTION_V1:
    display(step_table(EXECUTION_V1))
    print(json.dumps(gate_report(EXECUTION_V1), indent=2))
    print(json.dumps(guardrail_report(EXECUTION_V1), indent=2))

    if mlflow_tracking_server_uri:
        import mlflow

        mlflow.set_tracking_uri(mlflow_tracking_server_uri)
        runs = mlflow.search_runs(experiment_names=[mlflow_experiment_name],
                                  filter_string=f"tags.pipeline_execution_id = '{EXECUTION_V1.split('/')[-1]}'")
        cols = [col for col in ("tags.mlflow.runName", "status", "metrics.gate.primary_delta", "metrics.gate.primary_p_value",
                                "metrics.guardrail.violation_rate", "metrics.preferences.pairs") if col in runs.columns]
        display(runs[cols])
        print("MLflow UI:", sm.create_presigned_mlflow_tracking_server_url(
            TrackingServerName=mlflow_tracking_server_name, ExpiresInSeconds=300,
            SessionExpirationDurationInSeconds=1800)["AuthorizedUrl"])
    packages = sm.list_model_packages(ModelPackageGroupName="qwen3-0-6b-medical-rl", SortBy="CreationTime",
                                      SortOrder="Descending").get("ModelPackageSummaryList", [])
    for p in packages[:3]:
        print(p["ModelPackageArn"], p["ModelApprovalStatus"])

***
# Step 6: Trigger a re-run with new training data to validate automation

**Manual re-run.** Start the same definition with `InputDataS3Uri` pointing at `v2`. `BaselineEvaluation` reports a cache hit, because the benchmark and base model did not change. Every other step reruns on the new data.

**Automatic re-run.** An EventBridge rule starts the pipeline whenever a `_READY` marker object lands under `latest/`. The data team copies a new dataset version into `latest/` and writes the marker last. The rule targets the pipeline directly, so no Lambda function is needed. The rule turns on EventBridge notifications for the bucket and keeps any notification settings that already exist.

In [ ]:
EXECUTION_V2 = None
if RUN_BILLABLE:
    EXECUTION_V2 = start_execution("data-v2", InputDataS3Uri=DATA_V2, ModelVersion="v2", **GATE_OVERRIDES)
    time.sleep(60)
    display(step_table(EXECUTION_V2))  # BaselineEvaluation shows cache hit = True once it resolves
else:
    print("Skipped. Set RUN_BILLABLE = True to start the v2 execution.")

In [ ]:
rule_name = "qwen3-rl-pipeline-on-new-data"
events_role_name = "EventBridgeStartQwen3RlPipeline"
marker_key = f"{data_key}/latest/_READY"

if ENABLE_S3_TRIGGER:
    rl_pipeline.enable_bucket_eventbridge(s3, bucket_name)
    events_role_arn = rl_pipeline.ensure_events_role(boto3.client("iam"), events_role_name, pipeline_arn)
    time.sleep(10)  # let the new IAM role propagate before EventBridge uses it
    rule_arn = rl_pipeline.put_pipeline_rule(
        boto3.client("events"), rule_name,
        event_pattern={"source": ["aws.s3"], "detail-type": ["Object Created"],
                       "detail": {"bucket": {"name": [bucket_name]}, "object": {"key": [marker_key]}}},
        pipeline_arn=pipeline_arn, role_arn=events_role_arn,
        parameters={"InputDataS3Uri": DATA_LATEST, "ModelVersion": "auto-new-data", **GATE_OVERRIDES},
        description="Start the RL improvement pipeline when new training data is published",
    )
    print(rule_arn)
else:
    print("Skipped. Set ENABLE_S3_TRIGGER = True to create the rule.")

Publish new data the way a data team would: copy `v2` into `latest/`, then write the marker. Within a minute, a new execution whose display name is not one of yours appears in Studio and in the list below.

In [ ]:
if ENABLE_S3_TRIGGER and RUN_BILLABLE:
    for part in ("sft/train/dataset.json", "sft/test/dataset.json", "rl/dataset.json"):
        s3.copy_object(Bucket=bucket_name, Key=f"{data_key}/latest/{part}",
                       CopySource={"Bucket": bucket_name, "Key": f"{data_key}/v2/{part}"})
    s3.put_object(Bucket=bucket_name, Key=marker_key, Body=b"")
    time.sleep(90)
    for e in sm.list_pipeline_executions(PipelineName=pipeline_name, SortBy="CreationTime",
                                         SortOrder="Descending")["PipelineExecutionSummaries"][:5]:
        print(e["StartTime"].strftime("%H:%M:%S"), e["PipelineExecutionStatus"], e.get("PipelineExecutionDisplayName"),
              e["PipelineExecutionArn"].split("/")[-1])
else:
    print("Skipped. Needs ENABLE_S3_TRIGGER = True and RUN_BILLABLE = True, because this starts a full execution.")

***
## Save values for later labs

In [ ]:
RL_PIPELINE_NAME = pipeline_name
RL_PIPELINE_ARN = pipeline_arn
RL_PIPELINE_EVAL_DATA = EVAL_DATA
RL_PIPELINE_DATA_LATEST = DATA_LATEST
RL_PIPELINE_SFT_CONFIG = SFT_CONFIG_S3
%store RL_PIPELINE_NAME
%store RL_PIPELINE_ARN
%store RL_PIPELINE_EVAL_DATA
%store RL_PIPELINE_DATA_LATEST
%store RL_PIPELINE_SFT_CONFIG

## Clean up

Stop any running executions, then remove the trigger. Set `DELETE_PIPELINE = True` to delete the pipeline definition as well.

In [ ]:
DELETE_PIPELINE = False

for e in sm.list_pipeline_executions(PipelineName=pipeline_name)["PipelineExecutionSummaries"]:
    if e["PipelineExecutionStatus"] == "Executing":
        sm.stop_pipeline_execution(PipelineExecutionArn=e["PipelineExecutionArn"])
        print("stopping", e["PipelineExecutionArn"])

events = boto3.client("events")
try:
    events.remove_targets(Rule=rule_name, Ids=["rl-pipeline"])
    events.delete_rule(Name=rule_name)
    print("deleted rule", rule_name)
except events.exceptions.ResourceNotFoundException:
    pass

if DELETE_PIPELINE:
    sm.delete_pipeline(PipelineName=pipeline_name)
    print("deleted pipeline", pipeline_name)